---
#LangChain을 활용한 AI 자동화


---

# 학습 내용
- 환경 구축
- 간단한 챗봇 만들기, 메모리 추가하기
- Streamlit으로 챗봇 서비스 제작하기
- RAG 개요, Context Engineering의 기술
- RAG - PDF 문서 QA 챗봇 만들기
- RAG - 여러가지 문서 QA (PDF, DOC, EXCEL, PPT, CSV)
- RAG - 웹 페이지 QA
- RAG - 실시간 뉴스기사 QA
- 멀티모달 RAG







# 환경 구축


- 구글 드라이브 연동

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
%cd "/content/drive/MyDrive/업무/2026/인사교/헬스케어반(자연어처리)/agent/수업"

[Errno 2] No such file or directory: '/content/drive/MyDrive/업무/2026/인사교/헬스케어반(자연어처리)/agent/수업'
/content/drive/MyDrive/AI/인사교_LangChain_20260624


## 필수 라이브러리 설치

- LangChain 설치 : LangChain 1.0을 설치하면 langchain-core 등 필수 라이브러리가 함께 설치

In [23]:
!pip install -qU langchain>=1.0.0 langchain-community langchain-experimental langchain-text-splitters langchain_tavily

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


- 모델별 패키지 설치

In [27]:
!pip install -qU langchain-openai langchain-google-genai

- 벡터 DB 설치

In [28]:
# langchain-chroma, faiss-cpu : 벡터 DB를 오픈소스로 제공
# rank-bm25 : 전통적인 정보 검색 알고리즘으로, TF-IDF를 개선한 방식
!pip install -qU langchain-chroma faiss-cpu langchain-qdrant rank-bm25

- 구조화 관련 라이브러리 설치

In [26]:
# doc, excel, ppt, 구조화되지않는 문서를 다루는 라이브러리
# kiwipiepy : 한글 형태소 분석기
!pip install -q python-docx openpyxl python-pptx unstructured kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

- 라이브러리 가져오기

In [29]:
import os, time, base64
from glob import glob

# 문서 ID를 생성하기 위해 UUID 사용
from uuid import uuid4

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 모델 연동
import openai
from langchain.chat_models import init_chat_model
from langchain_openai.embeddings import OpenAIEmbeddings

# 프롬프트
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 메모리 기능
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# PDF 파일 로더
from langchain_core.documents import Document
from docx import Document as WordDocument
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredPowerPointLoader
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import OnlinePDFLoader

# 벡터 DB
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma
# LangChain에서 Qdrant 벡터 저장소 불러오기
from langchain_qdrant import QdrantVectorStore
# Qdrant 클라이언트 라이브러리 불러오기
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
#  특정 필터링 조건을 적용한 검색
from qdrant_client.http import models

# 웹 검색기
from langchain_community.tools import TavilySearchResults

# 청킹
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import MarkdownHeaderTextSplitter
# 지원되는 프로그램 언어 목록
from langchain_text_splitters import Language
from langchain_experimental.text_splitter import SemanticChunker

# 벡터 검색기
from langchain_community.retrievers import BM25Retriever

from kiwipiepy import Kiwi

# 마크다운 형태로 출력
from IPython.display import display, Markdown

#import google.generativeai as genai
from google import genai

In [5]:
import importlib.metadata

In [8]:
print(importlib.metadata.version('google-genai'))

2.10.0


- API 키 등록

In [34]:
with open("/content/drive/MyDrive/AI/인사교_LangChain_20260624/key/openai_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['OPENAI_API_KEY'] = api_key

# 간단한 챗봇 만들기

In [19]:
llm = init_chat_model(model="gpt-4o-mini")

chain = llm | StrOutputParser()

while True :
  q = input("🙂 ")

  if q == "exit" :
    break

  print("🤖 ", end="")
  print(chain.invoke(q))

🙂 ㅇㅇ
🤖 

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AQ.Ab8RN*****************************************DLWg. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

# 메모리 추가하기
- 리스트를 이용하여 간단하게 메모리 구현하기
   - 대화내용을 리스트에 저장하고 MessagesPlaceholder로 리스트 내용을 프롬프트에 추가해서 질문

In [ ]:
llm = init_chat_model(model="gpt-4o-mini")

# 챗팅 메시지를 저장하기 위한 것
chat_history = []

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 도우미 챗봇입니다."),
        # 과거의 대화내용(chat_history)을 저장하기 위한 메시지 공간을 설정
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

In [ ]:
chain = prompt | llm | StrOutputParser()

while True :
  q = input("🙂 ")

  if q == "exit" :
    break

  print("🤖 ", end="")
  # LLM으로 사용자 질문(input)과 과거 대화내용(chat_history)을 전송
  answer = chain.invoke({"input": q, "chat_history": chat_history})
  print(answer)

  # 현재 대화 내용을 리스트에 저장 (사용자 질문, AI 응답을 각각 튜플로 저장)
  chat_history.append(("human", q))
  chat_history.append(("ai", answer))

In [ ]:
chat_history

- RunnableWithMessageHistory를 이용한 메모리 구현

In [ ]:
llm = init_chat_model(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 도우미 챗봇입니다."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

chain = prompt | llm | StrOutputParser()

storage = {}

# 대화 메모리를 검색하는 함수 (세션id)
def get_session_history(session_id:str) -> ChatMessageHistory :
  # storage에 session_id가 없다면 (기존 챗팅 내용이 없다면)
  if session_id not in storage :
    # 새로 키를 만들어서 현재의 대화내용을 저장
    storage[session_id] = ChatMessageHistory()

  # 해당 세션 id의 대화 내용을 반환
  return storage[session_id]

memory_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    # 사용자 질문
    input_messages_key="input",
    # 대화 내용
    history_messages_key="chat_history"
)

# 세션 id 생성
session_id = str(uuid4())

while True :
  q = input("🙂 ")

  if q == "exit" :
    break

  print("🤖 ", end="")
  answer = memory_chain.invoke({"input": q},
                               config = {"configurable":{"session_id" : session_id}})
  print(answer)

In [ ]:
storage[session_id]

# Streamlit으로 챗봇 서비스 제작하기
  - 비개발 직군도 쉽게 활용하여 대시보드를 만들 수 있는 도구
  - 클라우드를 활용하여 결과물을 별다른 비용 없이 배포할 수 있는 서비스
  - 가장 빠르게 데이터 어플리케이션을 만들 수 있는 방법

  - 참고
    - https://docs.streamlit.io/library/get-started
    - https://github.com/MarcSkovMadsen/awesome-streamlit

- Streamlit 사용에 필요한 선수학습
  - 파이썬 사용 필요
  - Streamlit.io 및 Github 계정 필요
  - Github로 push할 수 있어야 함

- Streamlit의 장점

  - 간단하게 파이썬 코드로 앱을 빌드할 수 있음
  - 백엔드 개발이나 HTTP 요청 구현할 필요 없음
  - 커뮤니티에서 개발한 Component도 존재
  - Streamlit에서 배포할 수 있는 시스템 제공(신청 필요)
  - 화면을 녹화할 수 있는 Record 기능 제공
    - app을 빌드한 후, 오른쪽 ☰ 버튼을 클릭하면 Record a screencast를 확인할 수 있음

- 라이브러리 설치하기

In [ ]:
# pyngrok : 로컬 환경에서 실행 중인 서버를 외부 인터넷에서 접속 가능한 공개 URL로 즉시 연결해주는 Python 라이브러리
!pip install -qU streamlit pyngrok

- ngrok 토큰 등록
  - https://ngrok.com/ 에 로그인
  - 첫 화면에서 토큰 부분을 복사하여 메모장에 붙여넣고 key 폴더에 <font color=red>.ngrok_token</font> 파일명으로 저장
    - 파일형식을 모든 파일 (\*.\*)로 선택하여 .txt 확장자가 붙지않도록 함

  <pre>
  ngrok config add-authtoken <font color=red><b>1a2b3c4d5e6f7g8h9I0J</b></font>
  </pre>

In [ ]:
import os
with open("./key/ngrok_key.txt", "r") as f:
   api_key = f.read().strip()
os.system(f'ngrok config add-authtoken {api_key}')

- 실행함수

In [ ]:
import os, time
from pyngrok import ngrok

def run_streamlit_with_ngrok(app_file, port=8501, wait=10):
    # 1. 기존 프로세스 종료
    ngrok.kill()
    os.system("pkill -f streamlit")
    os.system("pkill -f ngrok")

    # 2. 새 앱 실행
    os.system(f"streamlit run {app_file} --server.port {port} --server.runOnSave true &")

    # 3. 서버가 뜰 때까지 대기
    time.sleep(wait)

    # 4. ngrok 새 연결
    public_url = ngrok.connect(port)
    print(f"✅ 실행 URL:", public_url)

    return public_url

- chat_message
  - 채팅 메시지 컨테이너 삽입
    - avatar : 채팅 메시지 왼쪽에 삽입되는 아이콘 설정

In [ ]:
%%writefile app01.py
import streamlit as st

# chat_input() : 챗팅 입력 컴포넌트
prompt = st.chat_input("대화 입력")

# AI 채팅 메시지
with st.chat_message("assistant", avatar="🤖") :
  # write() : 화면 해당 내용을 출력
  st.write("안녕하세요 AI 챗봇입니다")

# 대화 내용을 입력했다면
if prompt :
  with st.chat_message("user", avatar="😍") :
    st.write(prompt)

In [ ]:
run_streamlit_with_ngrok("app01.py")

- 채팅 페이지 제작

In [ ]:
%%writefile app02.py
import streamlit as st
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 타이틀 설정
st.title("✨ 나만의 챗봇 V1.0")

# 모델 설정
llm = init_chat_model(model="gpt-4o-mini", temperature=0, max_tokens=500)

# 챗 메시지 히스토리 초기화 (st.session_state)
if "messages" not in st.session_state :
  st.session_state.messages = []

# 챗봇을 시작할 때마다 기존 쳇 메시지를 갱신
for message in st.session_state.messages :
  # role : 대상 (assistant, user), content : 실제 대화내용
  with st.chat_message(message["role"]) :
    st.markdown(message["content"])

# 챗봇 기능 구현
# 사용자가 메시지를 입력했다면
# := : walrus 연산자, 바다코끼리 연산자
if prompt := st.chat_input("메시지 입력") :
  # 사용자 대화 내용을 session_state에 추가
  st.session_state.messages.append({"role" : "user",
                                    "content" : prompt})

  # 사용자 대화 내용 출력
  with st.chat_message("user", avatar="🤔") :
    st.markdown(prompt)

  # AI 응답 출력
  with st.chat_message("assistant", avatar="🤖") :
    result = llm.stream(prompt)
    response = st.write_stream(result)

  # AI 응답을 session_state에 추가
  st.session_state.messages.append({"role" : "assistant",
                                    "content" : response})

In [ ]:
run_streamlit_with_ngrok("app02.py")

- 실습 : 메모리 기능 추가

In [ ]:
%%writefile app03.py
import streamlit as st
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 메모리 관련 라이브리리
from uuid import uuid4
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 타이틀 설정
st.title("✨ 나만의 챗봇 V1.1")

# 메모리 설정
if "storage" not in st.session_state:
  st.session_state.storage = {}

# 메모리 검색 함수
def get_session_history(session_id:str) -> ChatMessageHistory :
  if session_id not in st.session_state.storage :
    st.session_state.storage[session_id] = ChatMessageHistory()

  return st.session_state.storage[session_id]

# 모델 설정
llm = init_chat_model(model="gpt-4o-mini", temperature=0, max_tokens=500)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 대화용 어시스턴트이다. 대화 맥락을 확인해서 응답을 해줘"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ]
)

chain = prompt | llm

memory_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# 챗 메시지 히스토리 초기화 (st.session_state)
if "messages" not in st.session_state:
    st.session_state.messages = []

# 챗봇을 시작할 때마다 기존 쳇 메시지를 갱신
for message in st.session_state.messages :
  # role : 대상 (assistant, user), content : 실제 대화내용
  with st.chat_message(message["role"]) :
    st.markdown(message["content"])

# 챗봇 내용을 저장하기 위한 session_id 생성
if "session_id" not in st.session_state:
    st.session_state.session_id = str(uuid4())

session_id = st.session_state.session_id

# 챗봇 기능 구현
# 사용자가 메시지를 입력했다면
# := : walrus 연산자, 바다코끼리 연산자
if user_input := st.chat_input("메시지 입력") :
  # 사용자 대화 내용을 session_state에 추가 (화면 출력용)
  st.session_state.messages.append({"role" : "user",
                                    "content" : user_input})

  # 사용자 대화 내용 출력
  with st.chat_message("user", avatar="🤔") :
    st.markdown(user_input)

  # AI 응답을 메모리에 등록하고 출력
  with st.chat_message("assistant", avatar="🤖") :
    response = memory_chain.stream({"input": user_input},
                                    config = {"configurable":{"session_id" : session_id}})

    final_response = st.write_stream(response)

  # AI 응답을 session_state에 추가 (화면 출력용)
  st.session_state.messages.append({"role" : "assistant",
                                    "content" : final_response})

In [ ]:
run_streamlit_with_ngrok("app03.py")

# RAG(Retrieval-Augmented Generation)
- <font color=red>기존의 LLM을 확장하여 주어진 컨텍스트나 질문에 대해 더욱 정확하고 풍부한 정보를 제공</font>하는 방법
- 즉, 모델이 학습 데이터에 포함되지 않은 외부 데이터를 실시간으로 검색(retrieval)하고 검색된 데이터를 활용(augmented)하여 이를 바탕으로 답변을 생성(generation)하는 것

- 기본 구조
  - <font color=red>검색 단계(Retrieval Phase)</font>: 사용자의 질문이나 컨텍스트를 입력으로 받아서, 이와 관련된 외부 데이터를 검색하는 단계
  - <font color=red>증강 단계(Augmented Phase)</font>: 검색된 데이터를 토큰화, 인코딩, 임베딩 후에 벡터 DB에 저장하고 검색기를 붙이는 단계  
  - <font color=red>생성 단계(Generation Phase)</font>: 벡터 DB에 저장된 데이터와 LLM 모델을 사용하여 사용자의 질문에 답변을 생성하는 단계

- 장점  
  - <font color=red>풍부한 정보 제공</font> : RAG 모델은 검색을 통해 얻은 외부 데이터를 활용하여, 보다 구체적이고 풍부한 정보를 제공
  - <font color=red>실시간 정보 반영</font> : 최신 데이터를 검색하여 반영함으로써, 모델이 실시간으로 변화하는 정보에 대응
  - <font color=red>환각 방지</font> : 검색을 통해 실제 데이터에 기반한 답변을 생성함으로써, 환각 현상이 발생할 위험을 줄이고 정확도 향상

- RAG 8단계 프로세스
  - 사전 준비 단계
    - 문서 가져오기
    - 텍스트 분할
    - 임베딩
    - 벡터 DB에 저장
  - Runtime 단계
    - 검색기 설정
    - 프롬프트 구성
    - LLM 모델 객체 생성
    - 체인 생성 및 실행

- RAG 패러다임의 발전 (https://futuredrill.stibee.com/p/46/)
  - 기본 RAG(Naive RAG) : **질문과 관련 있는 정보를 추가**함으로써 환각을 완화하는 기본적인 방식
  - 고급 RAG(Advanced RAG) : 검색 단계를 **사전 검색**과 **사후 검색**으로 나누어 검색 기법을 보완
  - 모듈형 RAG(Modular RAG) : **여러 가지 모듈들을 다양하게 조합**하는 방식으로 검색과 생성을 대체

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/langchain_rag01.png" width=60%>   
</center>

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/langchain_rag02.png" width=70%>   
</center>

## RAG의 발전 단계

<table>
<tr><td><b>구분</b><td><b>목적</b></td></td><td><b>명칭</b></td><td><b>설명</b></td></tr>
<tr><td>1세대</td>
<td>기본 RAG</td>
<td>Naive RAG</td>
<td>단순 검색 + LLM (질의→검색→상위k개 문서→LLM입력)</td></tr>
<tr><td rowspan=3>2세대</td>
<td rowspan=3>검색품질 강화</td>
<td>Advanced RAG</td>
<td>사전/사후 검색</td></tr>
<tr><td>TAG (Table-Augmented Generation)</td>
<td>테이블·구조화 데이터까지 포함</td></tr>
<tr>
<td>CAG (Cache-Augmented Generation)</td>
<td>자주 쓰는 질의·응답·검색 결과를 캐싱하여 속도·비용 절감</td></tr>
<tr><td rowspan=3>3세대</td>
<td rowspan=3>구조화·모듈화</td>
<td>Modular RAG</td>
<td>검색, 요약, 필터링, reasoning 모듈을 분리, 파이프라인 최적화 가능</td></tr>
<tr><td>Graph RAG</td>
<td>문서 간 관계를 Knowledge Graph로 변환 후 검색, 맥락 연결성 강화</td></tr>
<tr><td>Knowledge RAG</td>
<td>외부 검색뿐 아니라 온톨로지·도메인 지식 그래프를 통합, 지식베이스 중심 검색</td>
</tr>
<tr><td rowspan=2>4세대</td>
<td rowspan=2>추론·에이전트</td>
<td>Reasoning RAG</td>
<td>단순 검색+생성이 아니라, 검색한 정보를 기반으로 체계적 추론</td></tr>
<tr><td>Agentic RAG</td>
<td>RAG를 하나의 에이전트 구성, 다른 Agent와 협력 가능</td>
</tr>
<tr><td rowspan=3>5세대</td>
<td rowspan=3>상황 적응형·자기 개선</td>
<td>Adaptive RAG</td>
<td>질의 특성에 따라 검색 전략/매개변수 동적 변경</td></tr>
<tr><td>Corrective RAG</td>
<td>초기 생성 답변을 검증·수정, 오류 감지 → 재검색·재생성</td></tr>
<tr><td>Self RAG</td>
<td>LLM이 스스로 답변 품질을 평가하고 재검색, Self-Reflection 포함</td></tr>
</tr>
<tr><td rowspan=3>6세대</td>
<td rowspan=3>멀티모달 적용</td>
<tr><td>MultiModal RAG</td>
<td>텍스트·이미지·오디오·비디오를 함께 검색·생성</td>
</tr>
</table>


- 기본 RAG (Naive RAG)
  - ChatGPT의 광범위한 활용 이후 주목받기 시작한 초기 방법론을 대표
  - 전통적인 프로세스인 인덱싱, 검색, 생성으로 이루어짐

    - 인덱싱 (Indexing) : 외부 지식 데이터를 가져와 검색이 가능하도록 준비하는 단계
      - 원본 데이터를 정리하고 추출하여 다양한 파일 형식을 평문으로 변환
      - 텍스트를 더 작은 크기로 분할
      - 작게 나눈 텍스트들을 컴퓨터가 이해할 수 있도록 벡터로 변환

    - 검색 (Retrieve) : 사용자의 입력을 문서에 검색하여 정보를 검색하는 단계
       - 사용자의 입력을 기반으로, 내용을 벡터로 변환
       - 질문 임베딩과 위의 인덱싱 단계에서 작게 나눈 문서 블록 임베딩 간의 유사성을 계산
       - 유사성 수준에 따라 상위 K개의 문서 블록이 현재 질문에 대한 증강된 컨텍스트 정보로 선택
   - 생성 (Generation) : 증강된 정보를 원래 질문과 통합해 답변을 생성하는 단계
      - 주어진 질문과 관련 문서(2단계에서 선택된 상위 n개의 문서 블록)들은 새로운 프롬프트로 결합
      - 대형 언어 모델은 제공된 정보를 기반으로 질문에 답하는 임무를 수행  

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/langchain_rag03.jpg" width=60%>   
</center>  

## 현업에서 RAG 적용의 한계점
- 최신의 기술을 사용하여 현업에 적용하려면 다양한 문제에 부딪침 → 현업에 맞는 적절한 기술 적용 필요


<center>  
<img src="https://arome1004.cafe24.com/images/langchain/rag_problem02.png" width=60%>   
</center>  

- 데이터 전처리의 중요성
  - RAG는 자료를 기반으로 답변을 생성하기 때문에 자료 전처리가 RAG의 성능을 좌우 (**Garbage in - Garbage out**)
    - 영역구분 → 텍스트 추출 → 표 인식 → OCR → 메타 정보 → Tag 생성 (마이즈앤컴퍼니의 데이터 전처리 6단계)
  - 데이터 전처리가 어려운 이유
    - 입력 자료들에는 복잡한 표, 도식화된 이미지 등이 포함되어 있어 각각에 맞는 전처리가 필요

<center>  
<img src="https://arome1004.cafe24.com/images/langchain/doc_processing01.png" width=30%>   
</center>        

- RAG 개발 스택

<center>  
<img src="https://arome1004.cafe24.com/images/langchain/rag_problem03.png" width=60%><br><font size=1>출처 : https://www.analyticsvidhya.com/blog/2025/04/rag-developer-stack/</font>   
</center>  

- Open Source AI Stack

<center>  
<img src="https://arome1004.cafe24.com/images/langchain/rag_problem04.png" width=40%><br><font size=1>출처 : https://blog.bytebytego.com/p/ep146-the-open-source-ai-stack</font>   
</center>  

## RAG를 강화할 수 있는 영역들
- 쿼리 구조화
  - LLM를 활용하여 RDBMS, Cypher 쿼리 기반 그래프 DB, 벡터 DB를 검색할 수있도록 쿼리를 구조화하는 과정
- Query 변환
  - 쿼리 분해 : 복잡한 쿼리를 간단한 하위 쿼리로 분해하는 과정
    - **RAG-Fusion** : 쿼리생성 → 하위 쿼리 검색 → RRF(상호 순위 융합) 기반 검색문서 병합 및 순위 결합을 통해 가장 관련성 높은 쿼리 생성
  - 유사 문서 생성
    - **HyDE (가상 문서 임베딩)** : 검색 과정을 돕기 위해 질의에 기반하여 생성된 가상문서를 생성    

- 검색
  - Ranking : 관련성이 높은 순으로 검색 문서를 재정렬

- 현업에서는 상황에 맞도록 강화할 대상을 선택하고 단계적으로 진행해야 함
  
<center>  
<img src="https://arome1004.cafe24.com/images/langchain/rag_problem06.png" width=70%>
</center>

# Context Engineering의 기술
- 참고 : https://www.itworld.co.kr/article/4097715/더-많은-컨텍스트가-정답은-아니다-llm-최적화의-핵.html
- **LLM을 다루는 핵심 역량**으로, 실제로 AI 애플리케이션의 품질을 결정하는 요소는 모델이 응답을 생성할 때 접근할 수 있는 **정보 (컨텍스트)의 효과적인 작성** 기술
- 컨텍스트 엔지니어링은 단순한 기술적 보조 작업이 아니라 AI의 성능을 **평범함과
탁월함** 사이에서 갈라놓는 예술이자 과학으로 평가
- 컨텍스트란 단순히 가능한 한 **많은 정보를 프롬프트에 집어넣는 것이 아니라 기술적 제약 안에서 모델의 성능을 극대화**하기 위한 전략적 정보 아키텍처의 문제

## Context Window의 기술적 현실
- 최근의 LLM은 대략 8,000개에서 많게는 20만 개 이상의 토큰을 처리할 수 있는  컨텍스트 창을 지원
- 컨텍스트를 설계하고 활용 시 기술적 현실
  - **중간 정보 손실** : LLM은 긴 컨텍스트를 다룰 때 중간 부분의 정보에 대한 주의가 약화되는 경향
    - 모델은 컨텍스트 윈도의 앞부분과 뒷부분의 정보에 훨씬 더 민감하게 반응
    - 이는 트랜스포머 아키텍처가 순차 데이터를 처리하는 구조적 특성에서 비롯
  - **이론적 용량 vs 실제 처리 한계** : 모델이 모든 토큰을 동일한 정확도로 처리하는 것이 아님
    - 일반적으로 약 3만 2,000~6만 4,000 토큰을 넘어서면 정확도가 점차 감소하는 경향
    - 이는 인간의 작업 기억(Working Memory)과 비슷하게 머릿속에 많은 정보를 담을 수는 있지만, 실제로 효율적으로 사고하는 데에는 일부 핵심 정보에 집중하는 것이 더 효과적
  - **연산 비용** : 컨텍스트 길이는 지연 시간과 비용 모두에 제곱에 비례하여 증가
    - 예를 들어 1만 토큰짜리 컨텍스트를 10만 토큰으로 늘린다고 해서 비용이 단순히 10배가 아닌 최대 100배 이상 커질 수 있음
    - 비용을 사용자에게 전가하지는 않더라도, 컨텍스트가 길어질수록 연산 부담과 비용이 기하급수적으로 증가

## 컨텍스트 엔지니어링에서 얻은 교훈 4가지
- **최신성과 관련성이 양보다 중요**하다.
  - 실제 운영 환경에서 컨텍스트의 양을 줄이고, 대신 관련성과 최신성을 높였을 때 모델 성능이 눈에 띄게 개선 (AI CRM 스타트업 옥톨레인)
  - 예를 들어, G-mail에서 모든 이메일을 모델에 입력하는 것보다, 현재 진행 중인 거래와 의미적으로 연관된 이메일만 제공할 때 훨씬 더 정확한 결과
  - 컨텍스트가 너무 많으면 모델은 핵심 신호와 잡음을 구분하지 못함

- **콘텐츠만큼 구조도 중요**하다
  - LLM은 비구조적 데이터보다 구조화된 컨텍스트(XML 태그, 마크다운 헤더,구분선 등)에 훨씬 더 잘 반응

- **컨텍스트 계층화가 더 나은 검색**을 만든다.
  - 컨텍스트는 중요도와 관련성 기준으로 조직화
  - 핵심정보는 컨텍스트 창의 앞부분과 뒷부분에 배치하는 것이 가장 효과적
  - 최적의 구성 예시
    - 시스템 지시문 (시작 부분)
    - 현재 사용자 질의 (시작 부분)
    - 가장 관련성 높은 검색 정보 (시작 부분)
    - 보조 컨텍스트 (중간 부분)
    - 예시 및 예외 사례 (중간~후반부)
    - 최종 지시문 또는 제약 조건 (마지막 부분)

- **무상태(stateless)** 는 결함이 아니라 **기능** 이다
  - 모든 LLM 호출은 무상태(stateless)로 작동
  - 따라서 방대한 대화 이력을 그대로 유지하려 하기보다는, 똑똑한 컨텍스트 관리
방식을 도입하는 것이 효율적
    - 전체 대화 상태를 애플리케이션 내에서 관리
    - 요청마다 필요한 관련 이력만 모델에 전송
    - 시맨틱 청킹(semantic chunking)을 통해 중요한 정보만 선별
    - 대화 요약 기능을 도입해 긴 상호작용을 효율적으로 관리

## 프로덕션 시스템을 위한 실무 팁 7가지
- **시맨틱 청킹 구현**
  - 전체 문서를 그대로 모델에 전달하지 말고 의미 단위로 콘텐츠를 분할하고 임베딩하여 요청 시 관련 청크만 LLM에 전달
    <pre>
    Query → Embedding → Similarity 검색 → top-k chunks 검색 → Rerank (필요시) → Context 구조화 → LLM 호출
    </pre>

  - 일반적으로 컨텍스트 크기를 60-80% 줄이면서도 응답 품질을 20~30% 향상시키는 효과

- **단계적 컨텍스트 로딩**
  - 복잡한 질의에는 최소한의 컨텍스트로 시작하고, 필요할 때 단계적으로 추가
    - 첫 시도 : 핵심 지시문과 질의만 전달
    - 확신이 없을 때 : 관련 문서 추가
    - 여전히 불확실할 때 : 예시와 엣지 케이스 추가
  - 복잡한 질의에서도 품질을 유지하면서 평균 지연 시간과 비용을 감소

- **컨텍스트 압축 기술 활용**
  - 정보 손실 없이 컨텍스트를 줄이는 3가지 방법
    - **엔티티 추출** : 전체 문서 대신, 핵심 엔티티, 관계, 사실만 추출해 전송
    - **요약** : 과거 대화 내용의 요약본으로 대체 (요약 작업은 LLM를 활용)
    - **스키마 적용** : JSON, XML 등 구조화된 형식을 사용
  - 컨텍스트 창의 효율성을 높이는 동시에, 비용 절감과 응답 품질 유지라는 2가지 목표를 동시에 달성

- **컨텍스트 창 구현**
  - 대화형 시스템에서는 크기가 다른 sliding window를 유지해 컨텍스트를 효율적으로 관리
    - **즉시 창(immediate window)** : 최근 3~5회 발화. 전체 메시지를 원문 그대로 포함
    - **최근 창(recent window)** : 최근 10~20회 발화. 주요 요점을 요약한 형태로 저장
    - **히스토리 창(historical window)** : 그 이전 대화. 전체 대화의 주제를 간략히 정리한 고수준 요약만 유지   

- **스마트 캐싱 활용**
  - 컨텍스트를 캐싱 구조에 맞게 구성
  - **변하지 않는 부분**(시스템 지시문, 참조 문서 등)은 앞부분에 배치해 **캐시가 가능**하도록 하고, 자주 바뀌는 부분(사용자 질의, 검색된 컨텍스트 등)은 캐시 경계
이후에 배치
  - 반복되는 요청에서 입력 토큰 비용을 50~90%까지 절감

- **컨텍스트 활용도 측정**
  - 시스템이 컨텍스트를 얼마나 효율적으로 사용하고 있는지 측정하기 위해 모니터링 지표를 도입
    - 요청당 평균 컨텍스트 크기
    - 캐시 적중률
    - 검색된 컨텍스트의 관련성 점수
    - 컨텍스트 크기에 따른 응답 품질 변화
  - 데이터를 분석하면 과도한 컨텍스트 사용을 줄이고 효율성을 높일 수 있는 지점 도출
  - 많은 운영 시스템이 최적값보다 2~3배 더 많은 컨텍스트를 사용하고 있음

- **컨텍스트 초과 상황을 안정적으로 처리**
  - 모델의 컨텍스트 용량을 초과하는 상황이 발생할 때는 명확한 우선순위와 예외 처리 전략을 적용
    - 사용자 질의와 핵심 지시문을 최우선으로 유지
    - 앞뒤보다 덜 중요한 중간 구간을 먼저 제거
    - 자동 요약 기능을 활용해 초과된 부분을 압축 저장
    - 조용히 잘라내지 말고 명확한 오류 메시지를 반환해 상황을 알림
  - 정보 손실로 인한 품질 저하를 방지하고, 시스템이 예측 가능한 방식으로 컨텍스트 한계를 관리


## 고급 패턴 3가지
- **다중 턴 컨텍스트 관리**
  - 여러 차례 LLM 호출을 수행하는 에이전틱 시스템에서는 다중 턴 컨텍스트 관리가 핵심
  - 패턴 : 각 턴마다 누적되는 컨텍스트 누적기(Context Accumulator)를 유지하되, N번째 턴 이후에는 스마트 요약(Smart Summarization)을 적용해 컨텍스트가 무한히 커지는 것을 방지

- **컨텍스트 검색**
  - RAG 시스템에서는 다단계 검색을 구현
    - 관련 문서를 검색
    - 문서 내부에서 관련 섹션을 검색
    - 섹션 내부에서 관련 문단을 검색
  - 각 단계는 초점을 좁혀 관련성을 향상

- **컨텍스트 인지형 프롬프트 템플릿**
  - 사용 가능한 컨텍스트 크기에 따라 동적으로 적응하는 템플릿들을 작성  

## 피해야 할 패턴 5가지
- **전체 대화 이력을 그대로 전송** : 인사말, 확인 응답, 잡담 등 불필요한 내용까지 포함돼 토큰이 낭비
- **필터링 없이 데이터베이스 전체를 덤프** : 질의와 관련된 필드만 전송
- **모든 메시지에 동일한 지시문 반복** : 지속적인 규칙은 시스템 프롬프트나 캐시된 프리픽스를 활용
- **중간 정보 손실 현상을 무시** : 긴 컨텍스트의 중간에 핵심 정보를 배치하지 않음
- **최대 컨텍스트 창에 과도하게 의존** : 128K 토큰을 지원한다고 해서 항상 그만큼 사용할 필요는 없음

## 컨텍스트의 전망 기술
- **무한 컨텍스트 모델(Infinite Context Models)** : 검색증강을 활용해 사실상 무제한 길이의 컨텍스트를 처리하는 기술
- **컨텍스트 압축 모델(Context Compression Models)** : 주요 LLM에 전달하기 전에 컨텍스트를 압축·요약하는 특화 모델
- **학습 기반 컨텍스트 선택(Learned Context Selection)** : 질의에 최적화된 컨텍스트를 예측·선정하는 머신러닝 모델
- **멀티모달 컨텍스트(Multi-modal Context)** : 이미지, 오디오, 구조화 데이터 등을 자연스럽게 통합하는 차세대 컨텍스트 처리 방식

# RAG 1 :  PDF 문서 QA 챗봇 만들기
  - (1) 데이터 로드(Load Data)
    - RAG에 사용할 데이터를 불러오는 단계
  - (2) 텍스트 분할(Text Split)
    - 불러온 데이터를 작은 크기의 단위(chunk)로 분할하는 단계
  - (3) 임베딩 (Embedding) / 인덱싱 (Indexing)
    - 텍스트 데이터를 숫자로 이루어진 벡터로 변환하는 단계
    - 분할된 텍스트를 검색 가능한 형태로 만드는 단계
  - (4) 검색(Retrieval)
    - 사용자의 질문이나 주어진 컨텍스트에 가장 관련된 정보를 찾아내는 단계
  - (5) 생성(Generation)
    - 검색된 정보를 바탕으로 사용자의 질문에 답변을 생성하는 최종 단계

## 데이터 로드(Load Data)
  - 외부 데이터 소스에서 정보를 수집하고, 필요한 형식으로 변환하여 시스템에 로드
  - 가져온 데이터는 검색에 사용될 지식이나 정보를 담고 있어야 함
  - (예) 공개 데이터셋, 웹 크롤링을 통해 얻은 데이터, 또는 사전에 정리된 자료 등
  - 데이터 로드 라이브러리 종류
    - PDF : pypdf 라이브러리
    - HWP : langchain-teddynote의 HWPLoader
    - CSV : langchain_community의 CSVLoader
    - Excel : openpyxl 라이브러리
    - Word : docx2txt 라이브러리
    - PPT : python-pptx 라이브러리
    - 웹문서 : langchain_community의 WebBaseLoader
    - 텍스트 : langchain_community의 TextLoader
    - JSON : json 라이브러리

- PDF 파일 읽기

In [ ]:
loader = PyPDFLoader("./data/KDT.pdf")

documents = loader.load()

documents

- 디렉토리 로더 (DirectoryLoader)

In [ ]:
# data 폴더 내의 txt 파일만 링크만 가져온다
files = glob(os.path.join("./data", "*.txt"))
files

In [ ]:
loader = DirectoryLoader("./data", glob="*.txt", loader_cls=TextLoader)

documents2 = loader.load()

documents2[0]

## 텍스트 분할(Text Split)
  - <font color=red>CharacterTextSplitter</font> : 주어진 텍스트를 설정한 단위(문단(\n\n), 문장(\n), 단어(빈공백, 형태소) 등)로 분할
  - <font color=red>RecursiveCharacterTextSplitter</font> : 텍스트를 재귀적으로 분할하여 의미적으로 관련 있는 텍스트 조각들이 같이 있도록 하는 목적으로 설계
    - <font color=red>문자 리스트(['\n\n', '\n', ' ', ''])의 문자를 순서대로 사용하여 텍스트를 분할</font>하며, 분할된 청크들이 설정된 chunk_size보다 작아질 때까지 이 과정을 반복

  - <font color=red>split_documents()</font> : 데이터 로더의 반환값에서 page_content 항목을 찾아서 청크를 분리하는 함수
  - <font color=red>split_text()</font> : 데이터 로더의 반환값에서 page_content 항목만을 가져와서서 청크를 분리하는 함수
  - 반환값 구성
    - page_content : 분할된 토큰이 저장
    - metadata : 원본 문서에 대한 정보가 저장



- RecursiveCharacterTextSplitter를 이용한 텍스트 분리
  - 일반적인 텍스트에 권장되는 방식
  - 청크가 충분히 작아질 때까지 주어진 문자 목록의 순서대로 텍스트를 분할하려고 시도
  - 기본 문자 목록 : ["\n\n", "\n", " ", ""]
  - 단락 -> 문장 -> 단어 순서로 재귀적으로 분할

In [ ]:
# chunk_size : 분리할 문장의 문자수
# chunk_overlap : 이전 문장의 뒷부분을 다음 문장의 앞부분과 중복
#   -> 앞 문장과 뒷 문장 간의 연관성을 높이기 위해서
text_splitter2 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

text2 = text_splitter2.split_documents(documents)

text2

In [ ]:
print(len(text2))

for i, text in enumerate(text2) :
  print(text)
  print(f"분리문장 {i} : {len(text.page_content)}")

### [실습] 다양한 형태의 데이터 청킹 (선택)

- 코드 data chunking 하기

In [ ]:
# 지원되는 언어 목록 출력
print([e.value for e in Language])

In [ ]:
# Python 코드에 대한 기본적인 구분자 확인
RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)

In [ ]:
# 샘플 Python 코드
PYTHON_CODE = """
def hello_world():
    print("Hello, World!")

# Call the function
hello_world()
"""

# from_language() : Python 언어에 최적화된 text splitter 생성
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,  # 사용할 언어를 Python으로 설정
    chunk_size=50,             # 청크 크기를 50자로 설정
    chunk_overlap=0            # 청크 간 오버랩 없음
)
# Python 코드 문서를 분할하여 생성
python_docs = python_splitter.create_documents([PYTHON_CODE])
python_docs

- 마크다운 문서 Chunking

In [ ]:
markdown_text = """
# 🦜️🔗 LangChain

⚡ Building applications with LLMs through composability ⚡

## What is LangChain?

# Hopefully this code block isn't split
LangChain is a framework for...

As an open-source project in a rapidly developing field, we are extremely open to contributions.
"""
# LangChain의 RecursiveCharacterTextSplitter를 사용하여 Markdown 텍스트 분할
md_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,  # Markdown 언어 지정
    chunk_size=60,               # 청크 크기를 60자로 설정
    chunk_overlap=0              # 청크 간 오버랩 없음
)

# Markdown 문서를 청크로 나누기
md_docs = md_splitter.create_documents([markdown_text])
md_docs

In [ ]:
# Markdown 형식의 샘플 텍스트
markdown_document = """# Foo\n\n    ## Bar\n\nHi this is Jim\n\nHi this is Joe\n\n ### Boo \n\n Hi this is Lance \n\n
## Baz\n\n Hi this is Molly"""

# 헤더를 기준으로 Markdown을 분할하기 위한 규칙 설정
# 텍스트가 어느 계층에 포함되어 있는지 정보를 LLM 전달할 수 있다는 장점
headers_to_split_on = [
    ("#", "Header 1"),   # `#`은 Header 1로 분류
    ("##", "Header 2"),  # `##`는 Header 2로 분류
    ("###", "Header 3"), # `###`는 Header 3로 분류
]

# 헤더 기반으로 Markdown을 분할하는 Splitter 생성
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

# 설정한 헤더를 기준으로 Markdown을 분할
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

- Sementic Chunking
  - 임베딩 모델을 사용하여 텍스트의 의미(Semantic) 기반으로 청크를 생성해, 문맥적으로 연관된 내용을 하나의 청크로 묶는 방식

In [ ]:
# 임베딩 모델을 이용해서 의미적으로 청킹 분리
text_spliter3 = SemanticChunker(OpenAIEmbeddings())

text3 = text_spliter3.split_documents(documents)

print(len(text3))

for i, text in enumerate(text3) :
  #print(text)
  print(f"분리문장 {i} : {len(text.page_content)}")

## 임베딩 (Embedding)
  - 분리된 토큰을 인코딩하고 토큰간의 연관성으로 <font color=red>크기와 방향 성분 값으로 벡터화</font>하는 과정

<center>  
<img src="https://arome1004.cafe24.com/images/langchain/embedding.png" width=40%>   
</center>  











- 임베딩 결과 시각화 예시

<center>  
<img src="https://arome1004.cafe24.com/images/deeplearning/embedding1.png" width=60%>   
</center>

- 임베딩 (Embedding)의 필요성
  - 검색 시간을 단축시키고, 검색의 정확도를 높이는 데 중요한 역할
  - 텍스트 데이터를 벡터 공간 내에서 수학적으로 다룰 수 있음
  - 텍스트 간의 유사성 계산, 텍스트 데이터 기반 다양한 머신러닝 및 자연어 처리 작업 수행
  - 임베딩 과정은 텍스트의 의미적인 정보를 보존하도록 설계
  - 벡터 공간에서 가까이 위치한 텍스트 조각들은 의미적으로도 유사한 것으로 간주
  - 임베딩 모델 제공자
     - OpenAI의 OpenAIEmbeddings
     - Hugging Face의 HuggingFaceEmbeddings
     - Google의 GoogleGenerativeAIEmbeddings
     - OllamaEmbeddiongs (OpenSource) 등  

- 임베딩 실습 (OpenAIEmbeddings)

In [ ]:
# 모델 로드
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# 문서(문장) 리스트를 벡터로 변환 (임베딩)
text = [
        "안녕하세요",
        "반갑습니다",
        "당신은 이름은 무엇인가요?",
        "나의 친구는 나를 길동이라고 부른다",
        "안녕 길동!"
    ]

embeddings = embeddings_model.embed_documents(text)

# 임베딩된 문서 개수와 개별 임베딩 벡터의 차원 출력
len(embeddings), len(embeddings[0])

## 벡터 저장소

  - 벡터 형태로 표현된 데이터, 즉 임베딩 벡터들을 효율적으로 저장하고 검색할 수 있는 시스템이나 데이터베이스
  - <font color=red>임베딩 결과를 벡터DB에 저장 </font>
  - FAISS, Chroma, Pinecone, Qdrant 등


<center>  
<img src="https://arome1004.cafe24.com/images/langchain/vector_db.png" width=50%><br><font size=1>출처 : 벡터 DB 종류 (엔코아 블로그)</font>   
</center>  

#### **FAISS**
  - Facebook에서 만든 벡터 DB 라이브러리
  - 임시로 간단하게 벡터값을 저장하고 검색하는 기능을 제공
  - 대용량 벡터의 효율적인 검색 지원
  - 문서 검색, 추천 시스템, 이미지 검색 등에서 활용
  - 벡터 유사도를 계산하여 가장 가까운 문서를 빠르게 찾을 수 있음

In [ ]:
# 데이터 로딩
loader = PyPDFLoader("./data/KDT.pdf")

documents = loader.load()

In [ ]:
# 청킹
text_spliter3 = SemanticChunker(OpenAIEmbeddings())

text3 = text_spliter3.split_documents(documents)

In [ ]:
# 임베딩 -> 벡터 DB에 저장
vec_db = FAISS.from_documents(text3, OpenAIEmbeddings())

- FAISS 벡터 DB를 로컬에 저장하기

In [ ]:
vec_db.save_local("./db/faiss")

- 벡터 DB 불러오기

In [ ]:
# allow_dangerous_deserialization : pickle 파일을 읽을 수 있도록 설정 (역직렬화)
vec_db2 = FAISS.load_local("./db/faiss", OpenAIEmbeddings(),
                           allow_dangerous_deserialization=True)

- 검색 테스트

In [ ]:
results = vec_db2.similarity_search(query="KDT가 뭐야", k=1)

for r in results :
  print(r)

In [ ]:
results = vec_db2.similarity_search_with_score(query="KDT가 뭐야", k=1)

for doc, score in results :
  print(score)

- MMR (Maximum marginal relevance search)
  - 특정 쿼리에 대해 관련성이 높으면서도 서로 다양한 문서들을 선택하는 알고리즘
  - 검색 결과의 다양성과 관련성 사이의 균형을 맞추려고 시도

#### Chroma 벡터 DB

- 텍스트 검색과 추천 시스템 등에 활용
- Chroma는 쿼리 시 더 다양한 필터링과 조합이 가능

#### Qdrant 벡터 DB
- 실제 상용화에도 사용가능한 벡터 검색(Vector Search) 및 필터링을 지원하는 벡터 데이터베이스
- 메모리 내(in-memory) 실행 가능하며, 영구 저장(persistent storage)도 지원
- Web UI 지원 (https://cloud.qdrant.io/)
  - 로그인 후 Clusters를 생성
  - Client 설정정보 복사 (Python용)
- Hybrid 임베딩 지원 (키워드 기반, CoBERT 등)
- FAISS나 Chroma와 다르게, 메타데이터 기반 필터링 기능이 강력함

## 검색(Retrieval)
  - 사용자의 입력을 바탕으로 쿼리를 생성하고, 인덱싱된 데이터에서 가장 관련성 높은 정보를 검색

    - 벡터저장소 지원 검색기
    - 문맥 압축 검색기
    - 앙상블 검색기
    - 긴 문맥 재정렬
    - 상위 문서 검색기
    - 다중 쿼리 검색기
    - 셀프 쿼리 검색기
    - 시간 가중 벡터 저장소 검색기
    - 한글 형태소 분석기

### 벡터 스토어의 기본 검색기
- 검색기 설정 - 벡터 DB에서 제공하는 검색기 사용

In [ ]:
# 검색 수 설정
retriever = vec_db2.as_retriever(search_kwargs={"k":1})

- score_threshold : 쿼리 문장과 최소한 유사도 점수 이상인 문서만을 대상으로 추출
  - 유사도가 높은 문서만 필터링하고 싶을 때 유용

In [ ]:
retriever2 = vec_db1_back.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={'score_threshold': 0.3}
)

docs = retriever2.invoke(query)
print(len(docs))
docs[0]

- MMR 검색기 : 다양성 고려

In [ ]:
# k : 상위 5개 문서를 검색하여 반환
# fetch_k : 후보 집합으로 선택되는 문서의 수를 의미
# lambda_mult : 유사도(관련성)와 다양성 사이에서 적용될 수준을 의미 (0.5보다 작으면 다양성, 크면 관련성을 우선)
retriever3 = vec_db1_back.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'fetch_k': 50, 'lambda_mult': 0.15}
)

docs = retriever3.invoke(query)

print(len(docs))

docs[0]

- 메타데이터 필터링을 사용한 검색
  - 메타데이터의 특정 필드에 대해서 기준(예: 'format': 'PDF 1.4')을 설정하고 조건을 충족하는 문서만을 필터링하여 검색
  - 특정 형식이나 조건을 만족하는 문서를 검색할 때 유용

## 생성(Generation)
  - LLM 모델에 검색 결과와 함께 사용자의 입력을 전달하고 모델은 사전 학습된 지식과 검색 결과를 결합하여 주어진 질문에 가장 적절한 답변을 생성

  - LLM 선택
    - 오픈소스 모델 : DeepSeek, Llama, Mistral, Qween(한글 지원 우수), Gemma, GPT-oss 등
    - 상용 모델 : GPT, Claude, Gemini, Grok 등

  - 주요 파라미터
    - **temperature** : 0에 가까울수록 확정적, 1에 가까울수록 창의적 결과 생성
    - **top_p** : 확률 합이 top_p에 도달할 때까지 상위 토큰만 선택
    - **top_k** : 상위 확률 k개 토큰만 선택
    - **penalty_paramter** : 이미 생성된 토큰이 다시 등장할때 해당 토큰의 확률에 패널티 적용
    - **max_tokens** : 생성할 최대 토큰 수 지정 (대부분 LLM에서 사용)
    - **num_predict** : 생성할 목표 토큰 수 지정 (Mistral, LLaMA, Cohere, 일부 로컬 LLM 프레임워크에서 주로 쓰임)    
    - 로짓 생성 → penalty_paramter → temperature → top_k → top_p → 토큰 샘플링 → Stop 확인 순으로 적용됨


- 대화형 프롬프트 템플릿 설정 (아래 내용에서 불러다 쓰는 함수이므로 실행해야 함)

In [ ]:
results

In [ ]:
# 프롬프트 생성
template = """
     다음 문맥만을 참고해서 질문에 답해주세요 : {context}
     질문 : {question}

     **Source**를 기반으로 응답
"""

prompt = ChatPromptTemplate.from_template(template)

# document 파일을 텍스트(파일위치, 실제 검색 내용)로 변환
def format_docs(docs) :
  return "\n\n".join([f"""Source : {doc.metadata.get('source',
                  'Unknown')}\n{doc.page_content}""" for doc in docs])

- ChatGPT의 gpt-4o-mini를 사용하여 pdf를 학습하고 chain() 함수로 만들어줌

In [ ]:
llm = init_chat_model(model="gpt-4o-mini",
                      temperature = 0,
                      max_tokens=500)

chain = ({"context" : retriever | format_docs, "question" : RunnablePassthrough()}
         | prompt | llm | StrOutputParser())

In [ ]:
query = "K-디지털 트레이닝 아카데미 유형 명칭만 출력"

answer = chain.invoke(query)

print(answer)

### [실습] 다른 PDF 문서 내용을 대답해주는 챗봇 구현하기

In [ ]:
# 문서로드 (KHP_guide.pdf)
loader = PyPDFLoader("./data/KHP_guide.pdf")

documents = loader.load()
print(documents[0])
print(len(documents))

In [ ]:
# 청킹
text_spliter = SemanticChunker(OpenAIEmbeddings())

text = text_spliter.split_documents(documents)

print(len(text))

In [ ]:
# 임베딩 -> 벡터DB 저장
vec_db = FAISS.from_documents(text, OpenAIEmbeddings())

In [ ]:
# 로컬 폴더에 저장 및 불러오기
vec_db.save_local("./db/faiss2")

vec_db2 = FAISS.load_local("./db/faiss2", OpenAIEmbeddings(),
                           allow_dangerous_deserialization=True)

In [ ]:
# 검색기 설정
retriever = vec_db2.as_retriever(search_kwargs={"k":2})

In [ ]:
# 프롬프트 작성
template = """
     다음 문맥만을 참고해서 질문에 답해주세요 : {context}
     질문 : {question}

     **Source**를 기반으로 응답
"""

prompt = ChatPromptTemplate.from_template(template)

# document 파일을 텍스트(파일위치, 실제 검색 내용)로 변환
def format_docs(docs) :
  return "\n\n".join([f"""Source : {doc.metadata.get('source',
                  'Unknown')}\n{doc.page_content}""" for doc in docs])

In [ ]:
# 체인 생성
chain = ({"context" : retriever | format_docs, "question" : RunnablePassthrough()}
         | prompt | llm | StrOutputParser())

In [ ]:
# 체인 실행
answer = chain.invoke("KHP 사업계획서 최종 제출일자를 알려줘")
print(answer)

# RAG 2 : 여러가지 문서 QA (PDF, DOC, EXCEL, PPT, CSV)
- 참고 : 랭체인으로 RAG 4개발하기 : VectorRAG & GrpahRAG (길벗)

- 문서 (PDF, DOC, Excel, PPT, CSV) 읽기 함수

In [ ]:
#word 파일(.docx)에서 텍스트 추출
def load_docx(path) :
  doc = WordDocument(path)
  # doc.paragraphs : 문서 내 모든 문단을 반복할 수 있는 속성
  # 각 문단 사이 줄바꿈을 이용한 공백등이 있는 경우 제거
  result = [ p.text for p in doc.paragraphs if p.text.strip() != '' ]
  # 하나의 텍스트로 묶어서 반환
  return "\n".join(result)

In [ ]:
# Excel 파일(.xlsx)에서 텍스트 추출
def load_excel(path) :
  xls = pd.ExcelFile(path)
  all_sheets_texts = [] # 각 시트의 내용을 담을 리스트

  # 각 시트의 이름 가져오기
  for sheet_name in xls.sheet_names :
    # 각 시트를 DataFrame으로 변경
    df = xls.parse(sheet_name)
    # 각 시트의 내용을 하나로 결합
    # ravel() : 모델 셀의 값을 1차원 배열로 변환
    # map : 모든 셀 값을 str 함수를 이용해 문자열로 변환
    rs = "\n".join(map(str, df.values.ravel()))
    all_sheets_texts.append(rs) # 데이터리스트에 추가

  return "\n".join(all_sheets_texts) # 모든 시트내용을 하나로 결합하여 반환

In [ ]:
 # PDF 파일에서 텍스트 추출
def load_pdf(path):
    try:
        doc = PyPDFLoader(path).load()
        # 모든 페이지에서 텍스트(page_content) 추출하고
        # 각 페이지의 텍스트를 줄바꿈(\n)을 기준으로 결합하여 반환
        return "\n".join([page.page_content for page in doc])
    # PDF 파일이 손상되었거나 읽을 수 없을 경우 오류 메시지를 출력
    except Exception as e:
        print(f"Error processing PDF {path}. Error: {e}")
        return ""

In [ ]:
# CSV파일에서 텍스트 추출
def load_csv(path, column_name) :
  # CSV 파일 로더 생성
  loader = CSVLoader(
      file_path = path, # 불러올 CSV 파일 경로
      csv_args = {
          'delimiter' : ",", # 구분자 지정
          'quotechar' : '"', # 쌍따옴표를 하나의 묶인 값으로 처리
          'fieldnames' : column_name # 열 이름 지정
      }
  )
  docs = loader.load()
  return "\n".join([page.page_content for page in docs])

In [ ]:
# PPT 파일에서 텍스트 추출
def load_ppt(path) :
  loader = UnstructuredPowerPointLoader(
      path, # 읽어들일 파일의 경로
      mode = "elements", # 문서를 요소 단위로 나누어 처리(슬라이드 별로 제목,본문 등 분리)
      strategy = 'fast' # 속도 중심처리
  )
  docs = loader.load()
  return "\n".join([page.page_content for page in docs])

- 해당 폴더 내의 모든 파일 읽기

In [ ]:
all_texts = [] # 문서 유형에 상관없이 모두 담는 리스트

# 지정된 폴더 내 모든 파일과 하위 폴더를 탐색
for subdir, dirs, files in os.walk("./data/rag_files/") :
  for file in files :
    # 파일의 전체경로 생성
    filename = os.path.join(subdir,file)
    # 파일 확장자 추출
    extension = os.path.splitext(filename)[1].lower()
    text_content = "" # 추출한 파일내용을 담을 변수
    try :
      if extension == ".pdf" :
        text_content = load_pdf(filename)
      elif extension == ".docx" :
        text_content = load_docx(filename)
      elif extension in [".xls", ".xlsx"] :
        text_content = load_excel(filename)
      elif extension in [".ppt]", ".pptx"] :
        text_content = load_ppt(filename)
      elif extension == ".csv" : # 추후 필요한 컬럼명으로 변경
        text_content = load_csv(filename,
                                ["PassengeId", "Suvived", "Pclass", "Name", "Sex",
                              "Age", "Fare", "Cabin", "Embarked", "Famliy"])
    except Exception as e : # 예외처리
      print(f"파일 로딩 오류 : {filename}", f"Error : {e}")

    # 텍스트가 정상 추출일 경우 리스트에 추가
    if text_content :
      all_texts.append(text_content)

In [ ]:
all_texts[0]

- 텍스트를 임베딩하여 벡터 DB에 저장하고 불러옴

In [ ]:
# 모든 문서를 하나의 문자열로 합치기
combined_texts = "\n".join(all_texts)
# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
                                               chunk_overlap=50)
texts = text_splitter.split_text(combined_texts)
# OpenAI 임베딩 모델 로드
embeddings = OpenAIEmbeddings()
# FAISS 벡터 데이터베이스에 저장
vectorstore = FAISS.from_texts(texts, embeddings)
vectorstore.save_local("./db/faiss3")

In [ ]:
 # FAISS DB 로드
retriever = FAISS.load_local(
    './db/faiss3',
    OpenAIEmbeddings(),
    allow_dangerous_deserialization=True
).as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

- 체인 생성

In [ ]:
def format_docs(docs):
    return "\n\n".join([f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}" for doc in docs])

In [ ]:
template = """다음 문맥만을 참고하여 질문에 대답하세요.:
{context}

질문 : {question}

Source를 기반으로 응답 :"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
llm = init_chat_model(model="gpt-4o-mini", temperature=0)
chain = (
    {'context' : retriever | format_docs, 'question' : RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 엑셀 기반 질문
chain.invoke("마음 챙김의 리뷰는?")

In [ ]:
# CSV 기반의 질문
chain.invoke("승객이 탐승한 항구들은?")

# RAG 3 : 웹 페이지 QA
- 여러 웹사이트의 데이터 검색
- 참고 : 랭체인으로 RAG 4개발하기 : VectorRAG & GrpahRAG (길벗)

# RAG 4 : 실시간 뉴스기사 QA
- 뉴스기사의 내용에 대해 질문할 수 있는 뉴스기사 QA 앱 구축

- Tavily API를 사용하여 LangChain의 웹 검색 기능을 활용하기 위해서는 API 인증키가 필요
- https://tavily.com 에 접속하고 인증키를 발급 (월 1000회 무료 사용 가능)

In [ ]:
with open("./key/tavily_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['TAVILY_API_KEY'] = api_key

In [ ]:
# 웹 검색하기
query = "2026년 삼성의 주가 관련 뉴스를 검색해주세요."
# 도구생성
web_search = TavilySearchResults(max_results=10)
# 검색결과 반환
search_result = web_search.invoke(query)

In [ ]:
search_result

- 한 개의 문서로 통합하고 저장

In [ ]:
text = ""

for result in search_result :
  if result['score'] >= 0.6 : # 신뢰도에 대한 임계치 설정
    text += result['content']

In [ ]:
# 텍스트 파일로 저장
with open("./data/news.txt", 'w') as f:
  f.write(text)

In [ ]:
loader = TextLoader("./data/news.txt")
documents = loader.load()

In [ ]:
documents

- 텍스트 분리

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,
                                               length_function = len)
# 토큰화
texts = text_splitter.split_documents(documents)

- 임베딩 (Embedding)
- 벡터 저장소 저장 및 검색기 설정

In [ ]:
# 텍스트를 임베딩 후 DB 벡터에 저장
vec_db2 = Chroma.from_documents(documents=texts, embedding=OpenAIEmbeddings())

# 검색 기능 설정 (최대 2개의 결과 반환)
retriever = vec_db2.as_retriever(search_kwargs={"k": 2})

- 학습하고 chain() 함수로 만들어줌

In [ ]:
llm = init_chat_model(model="gpt-4o-mini", temperature=0)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
result = chain.invoke("2026년 삼성전자 주가 전망은 어떤가요?")
print(result)

# 멀티모달 RAG
- 텍스트, 이미지, 소리 등 다양한 형태의 데이터를 제공하는 RAG

In [20]:
def multimodal_rag(query, image_path) :
  # 이미지 파일을 바이너리 형태로 로드
  with open(image_path,"rb") as image_file :
    image_bytes = image_file.read()

  # GPT-4o 멀티모달 요청
  response = openai.chat.completions.create(
      model = "gpt-4o",
      messages = [
          {
              "role" : "system", "content" : """
              답변시 아래 양식을 준수하시오.

              안식번호 : 111가 1234

              다른 부가 설명이나 문장은 작성하지 마시오
              """},
          {
              "role" : "user",
              "content" : [
                  {"type" : "text", "text" : query},
                  {"type" : "image_url",
                    "image_url" : {"url" : "data:image/png;base64," +
                                   base64.b64encode(image_bytes).decode('utf-8')}
                  }
              ]
          }
      ]
   )
  return response.choices[0].message.content


In [33]:
query = "6월에 가장 적게 판매된 야채는 얼마나 팔렸나요?"
print(multimodal_rag(query,"/content/drive/MyDrive/AI/인사교_LangChain_20260624/data/chart.jpg"))

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AQ.Ab8RN*****************************************DLWg. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

- 자동자 번호판 인식

In [ ]:
query = "이미지에서 번호판을 인식해서 출력해주세요."
print(multimodal_rag(query,"./data/car_number.jpg"))

- 도로 표지판 인식

In [ ]:
query = "이미지에서 도로 표지 의미를 출력해주세요."
print(multimodal_rag(query,"./data/load_sign2.jpg"))

- 비디오(Video) 질의 응답 LLM (Gemini)
  - Goolgle API key 발급
    - https://aistudio.google.com/app/apikey?hl=ko
    - Gemini API와 달리, API key는 File API에 업로드한 데이터에 대한 접근 권한도 부여하므로 API key를 안전하게 보관하는 데 특별히 주의
  - File API를 사용하여 비디오 파일을 업로드
  - GenerateContent 요청을 통해 비디오에 대한 질문을 요청
  - 생성된 응답을 확인
  - 업로드한 Video 파일을 삭제

In [ ]:
cd /content/drive/MyDrive/AI/인사교_LangChain_20260624

In [11]:
import os

with open("/content/drive/MyDrive/AI/인사교_LangChain_20260624/key/google_api_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['GOOGLE_API_KEY'] = api_key

In [42]:
api_key

'AQ.Ab8RN6IX5eVbDdBE3c3d3jvYKcDTBWyylGOsGwTZJxq0tSDLWg'

In [36]:
 #비디오 파일 업로드
video_file_name = "/content/drive/MyDrive/AI/인사교_LangChain_20260624/golf.mp4"
print("파일을 업로드 중입니다...")

# 파일 업로드 및 파일정보 객체 반환
video_file = genai.upload_file(path=video_file_name)

# 업로드된 파일 URI 출력
print(f"업로드 URU : {video_file.uri}")

파일을 업로드 중입니다...


AttributeError: module 'google.genai' has no attribute 'upload_file'

In [12]:
client = genai.Client(api_key=api_key)

video_file_name = "/content/drive/MyDrive/AI/인사교_LangChain_20260624/golf.mp4"

file = client.files.upload(file=video_file_name)

In [13]:
print(f"업로드 URI : {file.uri}")

업로드 URI : https://generativelanguage.googleapis.com/v1beta/files/44l6jicy4yyd


- 파일을 업로드한 후, get()을 호출하여 API가 파일을 성공적으로 완료되었는지 확인 (실행하지 않아도 다음 내용은 수행됨)

In [14]:
#비디오 파일 처리 상태 확인
while file.state.name == "PROCESSING" :
  # 처리 중 메시지 출력
  print("비디오 업로드 및 전처리 완료까지 잠시만 기다려주세요..")
  time.sleep(5) # 5초 대기
  # 비디오 파일 상태 새로 조회하기
  file = client.files.get(name = file.name)

  # 처리 실패시 예외
  if file.state.name == "FAILED" :
    print("비디오 파일 처리 에러")

  # 처리 완료
  print("비디오 처리 완료. 다음 작업을 수행하세요")

비디이ㅗ 업로드 및 전처리 완료까지 잠시만 기다려주세요..
비디오 처리 완료. 다음 작업을 수행하세요


- 비디오가 업로드된 후, generate_content 함수를 사용하여 Video 에 대한 질문을 요청

In [22]:
response = client.models.generate_content(
    model = "gemini-2.5-flash", # 사용할 모델 이름
    contents = [
        "이 영상에 대해서 3줄 이내로 핵심만 요약해 주세요",
        file
    ]
)

In [25]:
print(response.text)

이 영상은 골프 테이크백과 코킹 잘하는 방법을 설명합니다.
어드레스 시 클럽과 몸의 공간을 유지하며 상체와 함께 클럽을 돌려야 합니다.
이때 손목이 누군가 누르는 듯한 느낌으로 자연스럽게 꺾이는(코킹) 것이 핵심입니다.


- 파일은 2일 후 자동으로 삭제되거나 delete()를 사용하여 수동으로 삭제

In [24]:
client.files.delete(name = file.name)

DeleteFileResponse(
  sdk_http_response=HttpResponse(
    headers=<dict len=10>
  )
)